<a href="https://colab.research.google.com/github/Vidit-01/pedestrian_intent_prediction/blob/yash/experiments/paper1_vlfm_jaad/notebooks/02_build_sample_manifest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:

from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

# Paths
JAAD_ROOT = Path("/content/JAAD")
SPLIT_ROOT = JAAD_ROOT / "split_ids" / "all_videos"
ANNOTATIONS_DIR = JAAD_ROOT / "annotations"
ATTRIBUTES_DIR = JAAD_ROOT / "annotations_attributes"

# Experiment output directory
EXPERIMENT_ROOT = Path(
    "/content/pedestrian_intent_prediction/experiments/paper1_vlfm_jaad"
)
RESULTS_DIR = EXPERIMENT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load the official video-level splits
video_split = {}

for split_name in ["train", "val", "test"]:
    split_file = SPLIT_ROOT / f"{split_name}.txt"

    if not split_file.exists():
        raise FileNotFoundError(f"Missing split file: {split_file}")

    for line in split_file.read_text(encoding="utf-8").splitlines():
        video_id = line.strip()
        if video_id:
            if video_id in video_split:
                raise ValueError(
                    f"{video_id} appears in multiple splits"
                )
            video_split[video_id] = split_name

print("Official split counts:")
print(pd.Series(video_split).value_counts())

# Parse behaviour-annotated pedestrian tracks
records = []

for xml_path in sorted(ANNOTATIONS_DIR.glob("*.xml")):
    video_id = xml_path.stem

    # Use only videos in the official default split.
    if video_id not in video_split:
        continue

    root = ET.parse(xml_path).getroot()

    attr_path = ATTRIBUTES_DIR / f"{video_id}_attributes.xml"
    attributes_by_id = {}

    if attr_path.exists():
        attr_root = ET.parse(attr_path).getroot()

        for element in attr_root.iter():
            track_id = element.attrib.get("id")
            if track_id:
                attributes_by_id[track_id] = element.attrib.copy()

    for track in root.findall(".//track"):
        if track.attrib.get("label") != "pedestrian":
            continue

        boxes = track.findall("box")
        if not boxes:
            continue

        first_box = boxes[0]
        track_id = first_box.findtext(
            "attribute[@name='id']"
        )

        if not track_id:
            track_id = track.attrib.get("id")

        if not track_id:
            continue

        attrs = attributes_by_id.get(track_id)

        # Only retain tracks with matching pedestrian attributes.
        if attrs is None:
            continue

        frame_numbers = [
            int(box.attrib["frame"])
            for box in boxes
            if "frame" in box.attrib
        ]

        if not frame_numbers:
            continue

        records.append({
            "video_id": video_id,
            "track_id": track_id,
            "split": video_split[video_id],
            "num_boxes": len(boxes),
            "first_frame": min(frame_numbers),
            "last_frame": max(frame_numbers),
            "track_crossing": attrs.get("crossing"),
            "crossing_point": attrs.get("crossing_point"),
            "decision_point": attrs.get("decision_point"),
            "label_protocol_verified": False,
        })

candidate_manifest = pd.DataFrame(records)

output_path = RESULTS_DIR / "jaad_candidate_tracks_all_videos.csv"
candidate_manifest.to_csv(output_path, index=False)

print("\nCandidate track count:", len(candidate_manifest))
print("\nTracks per split:")
print(candidate_manifest["split"].value_counts())

print("\nCrossing attribute distribution:")
print(
    candidate_manifest["track_crossing"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nCandidate tracks per split and crossing value:")
display(pd.crosstab(
    candidate_manifest["split"],
    candidate_manifest["track_crossing"],
    dropna=False
))

print("\nSaved candidate manifest to:", output_path)
display(candidate_manifest.head(10))


Official split counts:
train    188
test     126
val       32
Name: count, dtype: int64

Candidate track count: 686

Tracks per split:
split
train    341
test     292
val       53
Name: count, dtype: int64

Crossing attribute distribution:
track_crossing
-1    100
0      91
1     495
Name: count, dtype: int64

Candidate tracks per split and crossing value:


track_crossing,-1,0,1
split,,,
test,58,42,192
train,34,44,263
val,8,5,40



Saved candidate manifest to: /content/pedestrian_intent_prediction/experiments/paper1_vlfm_jaad/results/jaad_candidate_tracks_all_videos.csv


,video_id,track_id,split,num_boxes,first_frame,last_frame,track_crossing,crossing_point,decision_point,label_protocol_verified
0,video_0001,0_1_3b,train,569,0,568,-1,-1,363,False
1,video_0001,0_1_2b,train,70,0,69,-1,-1,26,False
2,video_0002,0_2_5b,val,192,18,209,1,53,43,False
3,video_0002,0_2_6b,val,158,52,209,1,81,73,False
4,video_0003,0_3_7b,train,175,0,174,1,-1,-1,False
5,video_0004,0_4_10b,train,119,0,118,0,0,33,False
6,video_0005,0_5_19b,test,180,0,179,-1,0,80,False
7,video_0005,0_5_16b,test,158,0,157,-1,-1,42,False
8,video_0005,0_5_14b,test,181,0,180,-1,-1,42,False
9,video_0005,0_5_18b,test,124,0,123,-1,-1,42,False


In [19]:

from pathlib import Path
import re

JAAD_CODE = Path("/content/JAAD/jaad_data.py")

if not JAAD_CODE.exists():
    raise FileNotFoundError(f"JAAD implementation not found: {JAAD_CODE}")

source_lines = JAAD_CODE.read_text(
    encoding="utf-8"
).splitlines()

# Locate methods related to crossing/intention sequence generation.
method_pattern = re.compile(
    r"^\s*def\s+.*(cross|intention|trajectory_sequence)",
    re.IGNORECASE
)

matches = [
    (i, line.strip())
    for i, line in enumerate(source_lines)
    if method_pattern.search(line)
]

print("Relevant JAAD method definitions:\n")
for line_number, definition in matches:
    print(f"Line {line_number + 1}: {definition}")

# Show the implementation around occurrences of the event markers.
for keyword in ["decision_point", "crossing_point"]:
    print(f"\n{'=' * 65}")
    print(f"Occurrences of {keyword}")
    print("=" * 65)

    locations = [
        i for i, line in enumerate(source_lines)
        if keyword in line
    ]

    if not locations:
        print("No occurrences found.")
        continue

    for i in locations:
        start = max(0, i - 12)
        end = min(len(source_lines), i + 18)

        print(f"\n--- Source lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1:4}: {source_lines[j]}")


Relevant JAAD method definitions:

Line 984: def generate_data_trajectory_sequence(self, image_set, **opts):
Line 1125: def _get_crossing(self, image_set, annotations, **params):
Line 1227: def _get_intention(self, image_set, annotations, **params):

Occurrences of decision_point

--- Source lines 463-492 ---
 463:                         'phone':list(int)
 464:                         'baby':list(int)
 465:                         'object':list(int)
 466:                         'stroller_cart':list(int)
 467:                         'bicycle_motorcycle':list(int)
 468:                     'attributes'(str): {
 469:                          'age': int
 470:                          'old_id': str
 471:                          'num_lanes': int
 472:                          'crossing': int
 473:                          'gender': int
 474:                          'crossing_point': int
 475:                          'decision_point': int
 476:                          'intersection': i

In [20]:

from pathlib import Path

source_path = Path("/content/JAAD/jaad_data.py")
lines = source_path.read_text(encoding="utf-8").splitlines()

# Print the complete _get_intention method.
start = next(
    i for i, line in enumerate(lines)
    if "def _get_intention(" in line
)

end = next(
    (
        i for i in range(start + 1, len(lines))
        if lines[i].startswith("    def ")
    ),
    len(lines)
)

print(f"_get_intention: lines {start + 1}-{end}")
for i in range(start, end):
    print(f"{i + 1:4}: {lines[i]}")


_get_intention: lines 1227-1310
1227:     def _get_intention(self, image_set, annotations, **params):
1228:         """
1229:         Generates intention data.
1230:         :param image_set: Data split to use
1231:         :param annotations: Annotations database
1232:         :param params: Parameters to generate data (see generade_database)
1233:         :return: A dictionary of trajectories
1234:         """
1235:         print('---------------------------------------------------------')
1236:         print("Generating intention data")
1237: 
1238:         num_pedestrians = 0
1239:         seq_stride = params['fstride']
1240:         sq_ratio = params['squarify_ratio']
1241:         height_rng = params['height_rng']
1242:         image_seq, pids_seq = [], []
1243:         box_seq, center_seq, occ_seq = [], [], []
1244:         intent_seq = []
1245:         video_ids, _pids = self._get_data_ids(image_set, params)
1246: 
1247:         for vid in sorted(video_ids):
1248:             i

In [21]:

from pathlib import Path
import inspect
import sys

JAAD_ROOT = Path("/content/JAAD")

if str(JAAD_ROOT) not in sys.path:
    sys.path.insert(0, str(JAAD_ROOT))

from jaad_data import JAAD

print("JAAD parser class loaded from:", inspect.getfile(JAAD))

source_path = JAAD_ROOT / "jaad_data.py"
lines = source_path.read_text(encoding="utf-8").splitlines()

# Inspect the public sequence-generation method and its defaults.
for i, line in enumerate(lines):
    if "def generate_data_trajectory_sequence(" in line:
        start = i
        break
else:
    raise RuntimeError("Could not find generate_data_trajectory_sequence")

end = next(
    (
        i for i in range(start + 1, len(lines))
        if lines[i].startswith("    def ")
    ),
    len(lines)
)

print(f"\nSequence-generation method: lines {start + 1}-{end}")
for i in range(start, end):
    print(f"{i + 1:4}: {lines[i]}")


JAAD parser class loaded from: /content/JAAD/jaad_data.py

Sequence-generation method: lines 984-1039
 984:     def generate_data_trajectory_sequence(self, image_set, **opts):
 985:         """
 986:         Generates pedestrian tracks
 987:         :param image_set: the split set to produce for. Options are train, test, val.
 988:         :param opts:
 989:                 'fstride': Frequency of sampling from the data.
 990:                 'sample_type': Whether to use 'all' pedestrian annotations or the ones
 991:                                     with 'beh'avior only.
 992:                 'subset': The subset of data annotations to use. Options are: 'default': Includes high resolution and
 993:                                                                                          high visibility videos
 994:                                                                            'high_visibility': Only videos with high
 995:                                                 

In [22]:

from pathlib import Path
import sys
import numpy as np

JAAD_ROOT = Path("/content/JAAD")
if str(JAAD_ROOT) not in sys.path:
    sys.path.insert(0, str(JAAD_ROOT))

from jaad_data import JAAD

jaad = JAAD(data_path=str(JAAD_ROOT))

configs = {
    "parser_defaults": {},
    "all_videos_beh": {
        "subset": "all",
        "sample_type": "beh",
        "data_split_type": "default",
        "min_track_size": 15,
        "fstride": 1,
    },
}

for config_name, opts in configs.items():
    print("\n" + "=" * 70)
    print("CONFIG:", config_name, opts or "(official defaults)")

    for split in ("train", "val", "test"):
        try:
            data = jaad.generate_data_trajectory_sequence(
                split,
                **opts
            )

            n_sequences = len(data["image"])
            lengths = [len(seq) for seq in data["image"]]
            label_values = sorted({
                int(label[0])
                for seq in data["intent"]
                for label in seq
            })

            print(
                f"{split}: sequences={n_sequences}, "
                f"label_values={label_values}, "
                f"sequence_length_min="
                f"{min(lengths) if lengths else None}, "
                f"sequence_length_max="
                f"{max(lengths) if lengths else None}"
            )
        except Exception as exc:
            print(f"{split}: ERROR {type(exc).__name__}: {exc}")



CONFIG: parser_defaults (official defaults)
---------------------------------------------------------
Generating action sequence data
fstride: 1
sample_type: all
subset: default
height_rng: [0, inf]
squarify_ratio: 0
data_split_type: default
seq_type: intention
min_track_size: 15
random_params: {'ratios': None, 'val_data': True, 'regen_data': False}
kfold_params: {'num_folds': 5, 'fold': 1}
---------------------------------------------------------
Generating database for jaad
Getting annotations for video_0001
Getting annotations for video_0002
Getting annotations for video_0003
Getting annotations for video_0004
Getting annotations for video_0005
Getting annotations for video_0006
Getting annotations for video_0007
Getting annotations for video_0008
Getting annotations for video_0009
Getting annotations for video_0010
Getting annotations for video_0011
Getting annotations for video_0012
Getting annotations for video_0013
Getting annotations for video_0014
Getting annotations for vide

In [23]:

from contextlib import redirect_stdout
from pathlib import Path
import io
import sys

JAAD_ROOT = Path("/content/JAAD")
if str(JAAD_ROOT) not in sys.path:
    sys.path.insert(0, str(JAAD_ROOT))

from jaad_data import JAAD

jaad = JAAD(data_path=str(JAAD_ROOT))

opts = {
    "subset": "all_videos",
    "sample_type": "beh",
    "data_split_type": "default",
    "min_track_size": 15,
    "fstride": 1,
}

results = {}

for split in ("train", "val", "test"):
    log = io.StringIO()

    try:
        with redirect_stdout(log):
            data = jaad.generate_data_trajectory_sequence(
                split, **opts
            )

        lengths = [len(seq) for seq in data["image"]]
        labels = [
            int(label[0])
            for seq in data["intent"]
            for label in seq
        ]

        results[split] = {
            "sequences": len(data["image"]),
            "frames_total": sum(lengths),
            "min_length": min(lengths) if lengths else None,
            "max_length": max(lengths) if lengths else None,
            "label_0_sequences": sum(
                int(seq[0][0]) == 0 for seq in data["intent"] if seq
            ),
            "label_1_sequences": sum(
                int(seq[0][0]) == 1 for seq in data["intent"] if seq
            ),
        }

    except Exception as exc:
        results[split] = {
            "error": f"{type(exc).__name__}: {exc}"
        }

for split, values in results.items():
    print(f"\n{split.upper()}")
    for key, value in values.items():
        print(f"  {key}: {value}")



TRAIN
  sequences: 321
  frames_total: 39727
  min_length: 15
  max_length: 579
  label_0_sequences: 32
  label_1_sequences: 289

VAL
  sequences: 52
  frames_total: 6031
  min_length: 22
  max_length: 860
  label_0_sequences: 7
  label_1_sequences: 45

TEST
  sequences: 283
  frames_total: 33801
  min_length: 16
  max_length: 430
  label_0_sequences: 58
  label_1_sequences: 225


In [24]:

from collections import Counter
from contextlib import redirect_stdout
import io
import re

video_pattern = re.compile(r"video_\d+")

for split in ("train", "val", "test"):
    with redirect_stdout(io.StringIO()):
        data = jaad.generate_data_trajectory_sequence(
            split,
            subset="all_videos",
            sample_type="beh",
            data_split_type="default",
            min_track_size=15,
            fstride=1,
        )

    counts = Counter()

    for sequence in data["image"]:
        if not sequence:
            continue

        match = video_pattern.search(str(sequence[0]))
        if match:
            counts[match.group(0)] += 1

    print(f"\n{split.upper()}")
    print("Unique videos with eligible sequences:", len(counts))
    print("Sequences represented:", sum(counts.values()))
    print("Videos contributing 1 sequence:", sum(n == 1 for n in counts.values()))
    print("Videos contributing multiple sequences:", sum(n > 1 for n in counts.values()))
    print("Maximum sequences from one video:", max(counts.values(), default=0))
    print("Videos with most sequences:", counts.most_common(5))



TRAIN
Unique videos with eligible sequences: 166
Sequences represented: 321
Videos contributing 1 sequence: 81
Videos contributing multiple sequences: 85
Maximum sequences from one video: 8
Videos with most sequences: [('video_0297', 8), ('video_0025', 6), ('video_0331', 6), ('video_0062', 5), ('video_0204', 5)]

VAL
Unique videos with eligible sequences: 30
Sequences represented: 52
Videos contributing 1 sequence: 16
Videos contributing multiple sequences: 14
Maximum sequences from one video: 5
Videos with most sequences: [('video_0274', 5), ('video_0340', 4), ('video_0006', 3), ('video_0172', 3), ('video_0306', 3)]

TEST
Unique videos with eligible sequences: 116
Sequences represented: 283
Videos contributing 1 sequence: 51
Videos contributing multiple sequences: 65
Maximum sequences from one video: 12
Videos with most sequences: [('video_0135', 12), ('video_0313', 11), ('video_0305', 9), ('video_0067', 8), ('video_0223', 8)]


In [25]:

from pathlib import Path
from contextlib import redirect_stdout
from collections import Counter
import hashlib
import io
import json
import re

import numpy as np
import pandas as pd
import sys

JAAD_ROOT = Path("/content/JAAD")
REPO_ROOT = Path("/content/pedestrian_intent_prediction")

EXP_ROOT = REPO_ROOT / "experiments" / "paper1_vlfm_jaad"
RESULTS_DIR = EXP_ROOT / "results"
CONFIGS_DIR = EXP_ROOT / "configs"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)

if str(JAAD_ROOT) not in sys.path:
    sys.path.insert(0, str(JAAD_ROOT))

from jaad_data import JAAD

jaad = JAAD(data_path=str(JAAD_ROOT))

OBSERVATION_FRAMES = 16
PREDICTION_HORIZON_FRAMES = 16
ASSUMED_FPS = 30

PARSER_OPTIONS = {
    "subset": "all_videos",
    "sample_type": "beh",
    "data_split_type": "default",
    "seq_type": "intention",
    "min_track_size": 15,
    "fstride": 1,
    "height_rng": [0, float("inf")],
    "squarify_ratio": 0,
}

print("Experiment root:", EXP_ROOT)
print("Parser options:", PARSER_OPTIONS)


Experiment root: /content/pedestrian_intent_prediction/experiments/paper1_vlfm_jaad
Parser options: {'subset': 'all_videos', 'sample_type': 'beh', 'data_split_type': 'default', 'seq_type': 'intention', 'min_track_size': 15, 'fstride': 1, 'height_rng': [0, inf], 'squarify_ratio': 0}


In [26]:

sequences_by_split = {}

for split in ("train", "val", "test"):
    log_buffer = io.StringIO()

    with redirect_stdout(log_buffer):
        sequences_by_split[split] = (
            jaad.generate_data_trajectory_sequence(
                split,
                **PARSER_OPTIONS,
            )
        )

    data = sequences_by_split[split]
    print(
        f"{split:5s}: "
        f"{len(data['image'])} sequences"
    )

print("\nParser sequence generation complete.")


train: 321 sequences
val  : 52 sequences
test : 283 sequences

Parser sequence generation complete.


In [27]:

def unwrap_scalar(value):
    """Unwrap nested single-item containers."""
    while isinstance(value, (list, tuple, np.ndarray)):
        if len(value) == 0:
            return None
        value = value[0]
    return value


def json_safe(value):
    """Convert nested NumPy values into JSON-serializable objects."""
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    return value


def find_video_id(image_ref):
    match = re.search(r"video_\d+", str(image_ref))
    if not match:
        return None
    return match.group(0)


manifest_rows = []
rejected_rows = []

required_length = (
    OBSERVATION_FRAMES + PREDICTION_HORIZON_FRAMES
)

for split, data in sequences_by_split.items():
    for seq_idx, image_sequence in enumerate(data["image"]):
        images = [str(x) for x in image_sequence]
        boxes = data["bbox"][seq_idx]
        labels = data["intent"][seq_idx]
        pids = data["pid"][seq_idx]

        n = len(images)

        # Check that all sequence-level arrays align.
        if not (len(boxes) == len(labels) == len(pids) == n):
            rejected_rows.append({
                "split": split,
                "sequence_index": seq_idx,
                "reason": "sequence_array_length_mismatch",
                "sequence_length": n,
            })
            continue

        if n < required_length:
            rejected_rows.append({
                "split": split,
                "sequence_index": seq_idx,
                "reason": "insufficient_frames_for_16_plus_16",
                "sequence_length": n,
            })
            continue

        video_id = find_video_id(images[0])
        pedestrian_id = str(unwrap_scalar(pids[0]))
        label_values = {
            int(unwrap_scalar(label))
            for label in labels
        }

        # Each parser intention sequence should have one constant label.
        if len(label_values) != 1:
            rejected_rows.append({
                "split": split,
                "sequence_index": seq_idx,
                "reason": "non_constant_track_label",
                "sequence_length": n,
            })
            continue

        label = next(iter(label_values))

        if label not in (0, 1):
            rejected_rows.append({
                "split": split,
                "sequence_index": seq_idx,
                "reason": "invalid_intention_label",
                "sequence_length": n,
            })
            continue

        if video_id is None:
            rejected_rows.append({
                "split": split,
                "sequence_index": seq_idx,
                "reason": "video_id_not_found_in_image_reference",
                "sequence_length": n,
                "first_image_ref": images[0],
            })
            continue

        # Last 32 frames: 16 observed + 16 subsequent.
        obs_start = n - required_length
        obs_end = obs_start + OBSERVATION_FRAMES
        target_start = obs_end
        target_end = target_start + PREDICTION_HORIZON_FRAMES

        observed_images = images[obs_start:obs_end]
        target_images = images[target_start:target_end]

        observed_boxes = boxes[obs_start:obs_end]
        target_boxes = boxes[target_start:target_end]

        # Verify that the window boundaries are correct.
        assert len(observed_images) == OBSERVATION_FRAMES
        assert len(target_images) == PREDICTION_HORIZON_FRAMES
        assert len(observed_boxes) == OBSERVATION_FRAMES
        assert len(target_boxes) == PREDICTION_HORIZON_FRAMES

        raw_id = f"{split}|{video_id}|{pedestrian_id}|{seq_idx}"
        sample_hash = hashlib.sha256(
            raw_id.encode("utf-8")
        ).hexdigest()[:12]

        manifest_rows.append({
            "sample_id": f"{split}_{sample_hash}",
            "split": split,
            "video_id": video_id,
            "pedestrian_id": pedestrian_id,
            "parser_sequence_index": seq_idx,
            "parser_sequence_length": n,

            "observation_start_position": obs_start,
            "observation_end_position_exclusive": obs_end,
            "target_start_position": target_start,
            "target_end_position_exclusive": target_end,

            "observation_frame_count": len(observed_images),
            "prediction_horizon_frame_count": len(target_images),

            "intention_label": label,
            "label_definition": (
                "JAAD official parser: 0 if crossing == -1; "
                "1 if crossing != -1, for behaviour tracks"
            ),

            "observation_image_refs_json": json.dumps(
                observed_images
            ),
            "target_window_image_refs_json": json.dumps(
                target_images
            ),
            "observation_bboxes_json": json.dumps(
                json_safe(observed_boxes)
            ),
            "target_window_bboxes_json": json.dumps(
                json_safe(target_boxes)
            ),

            "assumed_fps": ASSUMED_FPS,
            "sample_policy": (
                "One sample per eligible parser sequence; "
                "last 16 frames observed before final 16-frame segment"
            ),
        })


manifest = pd.DataFrame(manifest_rows)
rejected = pd.DataFrame(rejected_rows)

print("Candidate manifest rows:", len(manifest))
print("Rejected sequences:", len(rejected))

if not manifest.empty:
    print("\nSamples by split:")
    print(manifest["split"].value_counts().reindex(
        ["train", "val", "test"], fill_value=0
    ))

    print("\nLabels by split:")
    print(pd.crosstab(
        manifest["split"],
        manifest["intention_label"]
    ))

if not rejected.empty:
    print("\nRejection reasons:")
    print(rejected["reason"].value_counts())


Candidate manifest rows: 607
Rejected sequences: 49

Samples by split:
split
train    295
val       47
test     265
Name: count, dtype: int64

Labels by split:
intention_label   0    1
split                   
test             55  210
train            26  269
val               7   40

Rejection reasons:
reason
insufficient_frames_for_16_plus_16    49
Name: count, dtype: int64


In [28]:

assert not manifest.empty, (
    "Manifest is empty. Inspect the rejection reasons."
)

assert manifest["sample_id"].is_unique, (
    "Duplicate sample IDs found."
)

assert manifest["intention_label"].isin([0, 1]).all(), (
    "Unexpected label values."
)

assert (
    manifest["observation_frame_count"] == OBSERVATION_FRAMES
).all()

assert (
    manifest["prediction_horizon_frame_count"]
    == PREDICTION_HORIZON_FRAMES
).all()

assert (
    manifest["target_start_position"]
    == manifest["observation_end_position_exclusive"]
).all()

assert (
    manifest["target_end_position_exclusive"]
    <= manifest["parser_sequence_length"]
).all()

# A video must not occur in multiple splits.
video_split_counts = manifest.groupby(
    "video_id"
)["split"].nunique()

overlapping_videos = video_split_counts[
    video_split_counts > 1
]

print("Duplicate sample IDs: 0")
print("Invalid labels: 0")
print("Incorrect observation/horizon lengths: 0")
print("Videos appearing in multiple splits:", len(overlapping_videos))

if len(overlapping_videos):
    print(overlapping_videos)
    raise AssertionError(
        "Video leakage detected. Resolve before using this manifest."
    )

print("\nValidation checks passed.")
print("\nFinal manifest summary:")
print(
    manifest.groupby("split").agg(
        samples=("sample_id", "count"),
        unique_videos=("video_id", "nunique"),
        label_0=("intention_label", lambda s: (s == 0).sum()),
        label_1=("intention_label", lambda s: (s == 1).sum()),
    )
)


Duplicate sample IDs: 0
Invalid labels: 0
Incorrect observation/horizon lengths: 0
Videos appearing in multiple splits: 0

Validation checks passed.

Final manifest summary:
       samples  unique_videos  label_0  label_1
split                                          
test       265            112       55      210
train      295            158       26      269
val         47             27        7       40


In [29]:

manifest_path = RESULTS_DIR / "jaad_16x16_sample_manifest.csv"
rejected_path = RESULTS_DIR / "jaad_rejected_sequences.csv"
protocol_path = CONFIGS_DIR / "sample_protocol.json"

manifest.to_csv(manifest_path, index=False)
rejected.to_csv(rejected_path, index=False)

protocol = {
    "project": "Pedestrian Intention Prediction via Vision-Language Foundation Models",
    "dataset": "JAAD",
    "parser": "Official JAAD jaad_data.py",
    "parser_options": {
        "subset": "all_videos",
        "sample_type": "beh",
        "data_split_type": "default",
        "seq_type": "intention",
        "min_track_size": 15,
        "fstride": 1,
        "height_rng": [0, None],
        "squarify_ratio": 0,
    },
    "observation_frames": OBSERVATION_FRAMES,
    "prediction_horizon_frames": PREDICTION_HORIZON_FRAMES,
    "assumed_fps": ASSUMED_FPS,
    "window_policy": (
        "One sample per eligible parser sequence. The 16 observed "
        "frames immediately precede the final 16 frames of the "
        "retained intention sequence."
    ),
    "label_mapping": {
        "0": "JAAD parser assigns 0 when crossing == -1",
        "1": "JAAD parser assigns 1 when crossing != -1",
    },
    "model": {
        "name": "Qwen2.5-VL-3B-Instruct",
        "inference": "Local, zero-shot",
        "status": "Planned for Notebook 04",
    },
    "deviations_and_limitations": [
        "The local Qwen model substitutes for the paper's GPT-4V model.",
        "The exact paper sample-selection procedure has not been established.",
        "The one-sample-per-parser-sequence window policy is an explicit implementation choice.",
        "The parser's track-level intention label is used; no frame-level future labels are inferred.",
        "Vehicle-speed context and prompt optimization are not implemented in Notebook 02.",
        "Video frame availability and timestamp alignment must be verified before inference.",
    ],
    "counts": {
        split: int(count)
        for split, count in manifest["split"].value_counts().items()
    },
    "manifest_rows": int(len(manifest)),
    "rejected_sequence_rows": int(len(rejected)),
    "protocol_status": "candidate_manifest_created; paper_equivalence_not_verified",
}

protocol_path.write_text(
    json.dumps(protocol, indent=2),
    encoding="utf-8"
)

print("Saved manifest:", manifest_path)
print("Saved rejected-sequence report:", rejected_path)
print("Saved protocol:", protocol_path)
print("\nNotebook 02 candidate manifest created.")


Saved manifest: /content/pedestrian_intent_prediction/experiments/paper1_vlfm_jaad/results/jaad_16x16_sample_manifest.csv
Saved rejected-sequence report: /content/pedestrian_intent_prediction/experiments/paper1_vlfm_jaad/results/jaad_rejected_sequences.csv
Saved protocol: /content/pedestrian_intent_prediction/experiments/paper1_vlfm_jaad/configs/sample_protocol.json

Notebook 02 candidate manifest created.


In [30]:

columns_to_inspect = [
    "sample_id",
    "split",
    "video_id",
    "pedestrian_id",
    "parser_sequence_length",
    "observation_start_position",
    "target_start_position",
    "intention_label",
]

display(
    manifest[columns_to_inspect]
    .groupby("split", group_keys=False)
    .head(3)
    .reset_index(drop=True)
)

if len(manifest):
    example = manifest.iloc[0]

    print("\nExample sample:", example["sample_id"])
    print("Video:", example["video_id"])
    print("Pedestrian:", example["pedestrian_id"])
    print("Label:", example["intention_label"])

    print("\nObserved image references:")
    print(json.loads(example["observation_image_refs_json"]))

    print("\nTarget-window image references:")
    print(json.loads(example["target_window_image_refs_json"]))


,sample_id,split,video_id,pedestrian_id,parser_sequence_length,observation_start_position,target_start_position,intention_label
0,train_d2d2e6787a5b,train,video_0001,0_1_3b,364,332,348,0
1,train_00e83cc2b263,train,video_0003,0_3_7b,173,141,157,1
2,train_695daa3de06e,train,video_0004,0_4_10b,34,2,18,1
3,val_3b5cd5cda25d,val,video_0006,0_6_27b,201,169,185,1
4,val_cfc1536542b9,val,video_0006,0_6_32b,56,24,40,1
5,val_6cdb1185bee8,val,video_0021,0_21_92b,63,31,47,1
6,test_ba7d0938bc42,test,video_0005,0_5_12b,132,100,116,0
7,test_1d62f06af022,test,video_0005,0_5_13b,91,59,75,0
8,test_4ccb47e17f4c,test,video_0005,0_5_14b,43,11,27,0



Example sample: train_d2d2e6787a5b
Video: video_0001
Pedestrian: 0_1_3b
Label: 0

Observed image references:
['/content/JAAD/images/video_0001/00332.png', '/content/JAAD/images/video_0001/00333.png', '/content/JAAD/images/video_0001/00334.png', '/content/JAAD/images/video_0001/00335.png', '/content/JAAD/images/video_0001/00336.png', '/content/JAAD/images/video_0001/00337.png', '/content/JAAD/images/video_0001/00338.png', '/content/JAAD/images/video_0001/00339.png', '/content/JAAD/images/video_0001/00340.png', '/content/JAAD/images/video_0001/00341.png', '/content/JAAD/images/video_0001/00342.png', '/content/JAAD/images/video_0001/00343.png', '/content/JAAD/images/video_0001/00344.png', '/content/JAAD/images/video_0001/00345.png', '/content/JAAD/images/video_0001/00346.png', '/content/JAAD/images/video_0001/00347.png']

Target-window image references:
['/content/JAAD/images/video_0001/00348.png', '/content/JAAD/images/video_0001/00349.png', '/content/JAAD/images/video_0001/00350.png', 